In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sleepstudypilot/README.md
/kaggle/input/sleepstudypilot/SleepStudyData.csv
/kaggle/input/sleepstudypilot/SleepStudyPilot.csv


## Loading Dataset

In [2]:
df = pd.read_csv('/kaggle/input/sleepstudypilot/SleepStudyData.csv')

## Exporting necessary Libraries

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  OrdinalEncoder,OneHotEncoder,LabelEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

## Exploring Data

In [4]:
df.head()

,Enough,Hours,PhoneReach,PhoneTime,Tired,Breakfast
0,Yes,8.0,Yes,Yes,3,Yes
1,No,6.0,Yes,Yes,3,No
2,Yes,6.0,Yes,Yes,2,Yes
3,No,7.0,Yes,Yes,4,No
4,No,7.0,Yes,Yes,2,Yes


In [5]:
#Enough = Do you think that you get enough sleep?
# Hours = On average, how many hours of sleep do you get on a weeknight?
# PhoneReach = Do you sleep with your phone within arms reach?
# PhoneTime = Do you use your phone within 30 minutes of falling asleep?
# Tired = On a scale from 1 to 5, how tired are you throughout the day? (1 being not tired, 5 being very tired)
# Breakfast = Do you typically eat breakfast?

In [6]:
df.shape

(104, 6)

# Handle Missing Values

## Identify Missing Values

In [7]:
df.isnull().sum()

Enough        0
Hours         2
PhoneReach    0
PhoneTime     0
Tired         0
Breakfast     0
dtype: int64

## Identify numerical and categorical columns

In [8]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

## Imputing Missing Values

In [9]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
df.isnull().sum()

Enough        0
Hours         0
PhoneReach    0
PhoneTime     0
Tired         0
Breakfast     0
dtype: int64

# Dropping Uncessary Column

In [11]:
df = df.drop(columns=['PhoneReach'])

# Encoding Categorical Variables

## Identify Categorical Columns

In [12]:
cat_cols = df.select_dtypes(include=['object']).columns

print("Categorical columns:", cat_cols)

Categorical columns: Index(['Enough', 'PhoneTime', 'Breakfast'], dtype='object')


In [13]:
for col in cat_cols:
    unique_values = df[col].unique()
    print(f"Unique values in '{col}':", unique_values)

Unique values in 'Enough': ['Yes' 'No']
Unique values in 'PhoneTime': ['Yes' 'No']
Unique values in 'Breakfast': ['Yes' 'No']


## For Ordinal Categories

In [14]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_cols = ['Tired']  # Example of ordinal column
ordinal_encoder = OrdinalEncoder(categories=[['1', '2', '3', '4', '5']])
df['Tired'] = ordinal_encoder.fit_transform(df[['Tired']])

## For Nominal Categories

In [15]:
selected_cat_cols = ['Enough', 'PhoneTime', 'Breakfast']

df[selected_cat_cols] = df[selected_cat_cols].astype(str)

onehot_encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
onehot_encoded = onehot_encoder.fit_transform(df[selected_cat_cols])

onehot_encoded_df = pd.DataFrame(onehot_encoded, columns=onehot_encoder.get_feature_names_out(selected_cat_cols))

df = df.drop(selected_cat_cols, axis=1).reset_index(drop=True)
onehot_encoded_df = onehot_encoded_df.reset_index(drop=True)

df = pd.concat([df, onehot_encoded_df], axis=1)

df.head()

/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


,Hours,Tired,Enough_No,Enough_Yes,PhoneTime_No,PhoneTime_Yes,Breakfast_No,Breakfast_Yes
0,8.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0
1,6.0,2.0,1.0,0.0,0.0,1.0,1.0,0.0
2,6.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
3,7.0,3.0,1.0,0.0,0.0,1.0,1.0,0.0
4,7.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0


# Feature Scaling

## Min-Max Scaling

In [16]:
scaler = MinMaxScaler()
df[['Tired']] = scaler.fit_transform(df[['Tired']])
print(df)


     Hours  Tired  Enough_No  Enough_Yes  PhoneTime_No  PhoneTime_Yes  \
0      8.0   0.50        0.0         1.0           0.0            1.0   
1      6.0   0.50        1.0         0.0           0.0            1.0   
2      6.0   0.25        0.0         1.0           0.0            1.0   
3      7.0   0.75        1.0         0.0           0.0            1.0   
4      7.0   0.25        1.0         0.0           0.0            1.0   
..     ...    ...        ...         ...           ...            ...   
99     7.0   0.25        1.0         0.0           0.0            1.0   
100    7.0   0.50        1.0         0.0           0.0            1.0   
101    8.0   0.50        0.0         1.0           0.0            1.0   
102    7.0   0.25        0.0         1.0           0.0            1.0   
103    6.0   0.50        0.0         1.0           0.0            1.0   

     Breakfast_No  Breakfast_Yes  
0             0.0            1.0  
1             1.0            0.0  
2             0.0 

# Train Test Split

In [17]:
X = df.drop('Tired', axis=1) 
y = df['Tired']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set shape: X={X_train.shape}, y={y_train.shape}')
print(f'Testing set shape: X={X_test.shape}, y={y_test.shape}')

Training set shape: X=(83, 7), y=(83,)
Testing set shape: X=(21, 7), y=(21,)
